Poissonov problem $\\$
$-u''(x) = f(x)$ na $(0,1)$ $\\$
$u(0)=u(1)=0$

In [ ]:
from netgen.meshing import Mesh as NGMesh, MeshPoint, Element1D, Element0D, Pnt
from ngsolve import *

import scipy.sparse as sp
from scipy.sparse import csr_matrix
import matplotlib.pylab as plt
import numpy as np

In [ ]:
unit_interval = NGMesh(dim=1)

N = 40
pnums = []

for i in range(N+1):
    xc = i / N
    pnums.append(unit_interval.Add(MeshPoint(Pnt(xc, 0, 0))))

# 1D domain region
idx_dom = unit_interval.AddRegion("omega", dim=1)
for i in range(N):
    unit_interval.Add(Element1D([pnums[i], pnums[i+1]], index=idx_dom))

# 0D boundary regions
idx_left  = unit_interval.AddRegion("left", dim=0)
idx_right = unit_interval.AddRegion("right", dim=0)

unit_interval.Add(Element0D(pnums[0],  index=idx_left))
unit_interval.Add(Element0D(pnums[-1], index=idx_right))

mesh = Mesh(unit_interval)

In [ ]:
print("dim =", mesh.dim)
print("vertices =", mesh.nv)
print("elements =", mesh.ne)
print("boundaries =", mesh.GetBoundaries())
print("materials =", mesh.GetMaterials())

In [ ]:
fes = H1(mesh, order=1, dirichlet="left|right")

In [ ]:
print ("ndof =", fes.ndof)

In [ ]:
u = fes.TrialFunction()
v = fes.TestFunction()

f = LinearForm(fes)
f += x*(1-x)*v*dx

a = BilinearForm(fes)
a += InnerProduct(grad(u), grad(v)) * dx

a.Assemble()
f.Assemble()

In [ ]:
n = a.mat.height
density = a.mat.nze / (n*n)
print("Ukupan broj elemenata:", n*n)
print("Broj nenul elemenata:", a.mat.nze)
print("Popunjenost:", density)

In [ ]:
plt.rcParams['figure.figsize'] = (12, 12)
A = sp.csr_matrix(a.mat.CSR())

plt.spy(A)
plt.show()

In [ ]:
print (csr_matrix(a.mat.CSR()))

In [ ]:
print (f.vec)

In [ ]:
solution_gf = GridFunction(fes)
solution_gf.vec.data = a.mat.Inverse(fes.FreeDofs()) * f.vec

In [ ]:
print(solution_gf.vec)

In [ ]:
exact = lambda x: x*(x - 1)*(x*x - x - 1)/12

In [ ]:
x_points = np.linspace(0, 1, 2000)
solution_np = np.array([solution_gf(xi) for xi in x_points])
exact_np = np.array([exact(xi) for xi in x_points])

plt.rcParams.update({'font.size': 18})

plt.plot(x_points, exact_np, linewidth=6, color='#4285F4', label="Exact")
plt.plot(x_points, solution_np, linewidth=4, linestyle='-.', color='#DB4437', label="FEM")
plt.xlabel("x")
plt.ylabel("u_h(x)")
plt.title("1D FEM solution")
plt.grid()
plt.legend()
plt.legend(frameon=True)
plt.show()

In [ ]:
exact_cf = x*(x - 1)*(x*x - x - 1)/12

error_L2 = sqrt(Integrate((solution_gf - exact_cf)**2, mesh))
print ("L2 norm error:", error_L2)

In [ ]:
exact_grad_cf = exact_cf.Diff(x)
error_H1_semi = sqrt(Integrate((grad(solution_gf) - exact_grad_cf)**2, mesh))
print ("H1 seminorm error:", error_H1_semi)


In [ ]:
error_H1 = sqrt(error_L2**2 + error_H1_semi**2)
print ("H1 norm error:", error_H1)